# Self-Supervised Learning

In [3]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import torch
from torch import nn
import copy
from src.load import PROCESSED_DIR, MODEL_DIR, make_loader

In [2]:
data = np.load(PROCESSED_DIR / 'splits.npz')
X_train = data['X_train']
print(X_train.shape)

train_loader_ssl = make_loader(X_train, batch_size=128, shuffle=True)

(14525, 6, 300)


## Autoencoder

In [7]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(6, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            
            nn.Conv1d(32, 64, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.Conv1d(64, 128, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU()
        )

    def forward(self, x):
        return self.cnn(x)


class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.ConvTranspose1d(128, 64, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            
            nn.ConvTranspose1d(64, 32, kernel_size=5, stride=2, padding=2, output_padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.ConvTranspose1d(32, 6, kernel_size=5, stride=2, padding=2, output_padding=1)
        )

    def forward(self, x):
        return self.cnn(x)

    
class AutoEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

In [4]:
def train_autoencoder(loader, epochs=10, lr=1e-3, device="cpu"):
    model = AutoEncoder().to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    best_loss = float("inf")
    best_state = None

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        total_samples = 0

        for X in loader:
            X = X.to(device)
            optimizer.zero_grad()

            X_hat = model(X)
            loss = criterion(X_hat, X)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * X.size(0)
            total_samples += X.size(0)

        avg_loss = total_loss / total_samples
        print(f"Epoch {epoch+1} | Loss: {avg_loss:.6f}")

        if avg_loss < best_loss:
            best_loss = avg_loss
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)

    return model.encoder

In [6]:
encoder = train_autoencoder(train_loader_ssl, epochs=10, lr=1e-3)
torch.save(encoder.state_dict(), MODEL_DIR / 'encoder_ae.pth')

Epoch 1 | Loss: 0.275314
Epoch 2 | Loss: 0.073013
Epoch 3 | Loss: 0.049388
Epoch 4 | Loss: 0.038865
Epoch 5 | Loss: 0.033481
Epoch 6 | Loss: 0.029175
Epoch 7 | Loss: 0.025857
Epoch 8 | Loss: 0.023384
Epoch 9 | Loss: 0.021880
Epoch 10 | Loss: 0.020648


## Evaluate

In [ ]:
import torch
import torch.nn as nn

def evaluate_autoencoder(model, loader, device="cpu"):
    model.eval()
    criterion = nn.MSELoss()
    total_loss = 0
    total_samples = 0

    with torch.no_grad():
        for X in loader:
            X = X.to(device)
            X_hat = model(X)
            
            if X_hat.shape[-1] != X.shape[-1]:
                X_hat = X_hat[:, :, :X.shape[-1]]
                
            loss = criterion(X_hat, X)
            
            total_loss += loss.item() * X.size(0)
            total_samples += X.size(0)

    avg_loss = total_loss / total_samples
    print(f"Evaluation MSE Loss: {avg_loss:.6f}")
    return avg_loss

In [4]:
# load test set
data = np.load(PROCESSED_DIR / 'splits.npz')
X_test = data['X_test']
test_loader = make_loader(X_test, batch_size=128, shuffle=False)
print(X_test.shape)

(4150, 6, 300)


In [ ]:
encoder = AutoEncoder().encoder
encoder.load_state_dict(torch.load(MODEL_DIR / 'encoder_ae.pth'))
test_loss = evaluate_autoencoder(encoder, test_loader)